# YOLOv8s 训练 - Block 检测

## 操作步骤

### 第一步：上传数据集到 Google Drive
1. 先用 `prepare_dataset.py` 处理好数据集（标注 → 划分 → 打包）
2. 浏览器打开 https://drive.google.com/
3. 把本地的 `dataset_switch.zip` **拖拽上传**到 Drive 根目录
4. 等上传完成（Drive 支持断点续传）

### 第二步：在 Colab 中运行以下单元格
按顺序点击每个单元格左边的 ▶ 运行即可

In [ ]:
# 1. 安装依赖
!pip install ultralytics -q
from ultralytics import YOLO

In [ ]:
# 2. 从 Google Drive 加载数据集

# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 把 dataset_switch.zip 上传到 Google Drive 根目录后执行：
!cp /content/drive/MyDrive/dataset_switch.zip /content/

# 解压
!unzip -qo dataset_switch.zip -d /content/dataset/

# 检查
import os
if os.path.exists('/content/dataset/data.yaml'):
    print(f'✅ 成功！训练集: {len(os.listdir("/content/dataset/images/train"))} 张')
    print(f'   验证集: {len(os.listdir("/content/dataset/images/val"))} 张')
else:
    print('❌ 解压失败，请检查 dataset_switch.zip 是否在 Google Drive 根目录')
    !ls -la /content/

In [ ]:
# 3. 训练 YOLOv8s 模型（block 检测）
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用设备: {device}')

model = YOLO('yolov8s.pt')
model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=8 if device=='cpu' else 16,
    device=device,
    workers=2,
    patience=30,
    name='block_20260622',     # 换个名字，避免和旧 best.pt 混淆
    exist_ok=False,
)

In [ ]:
# 4. 下载训练好的模型
from google.colab import files
files.download('/content/runs/detect/block_20260622/weights/best.pt')